In [ ]:
"""
================================================================================
LAB: Student Record Management System
File: Day1_Lab_RecordManager_starter.py
--------------------------------------------------------------------------------
Trainer Section (First 30%):
  - In-memory database initialization
  - File loading logic with JSON exception handling
  - View all records formatted output
  - Interactive CLI loop scaffold

Student Completion Tasks (Remaining 70%):
  1. Complete add_student_record() with input validation
  2. Implement search_student_record() by ID or Name substring
  3. Implement delete_student_record() with confirmation
  4. Implement update_student_record()
  5. Implement save_records_to_json() with safe file flushing
  6. STRETCH GOAL: Implement export_to_csv()
================================================================================
"""

import json
import os
import sys
from typing import Dict, Any

DATABASE_FILE = "sample_records.json"


STUDENT_REGISTRY: Dict[str, Dict[str, Any]] = {}


def load_records_from_json(file_path: str) -> Dict[str, Dict[str, Any]]:

    if not os.path.exists(file_path):
        print(f"[WARN] Database file '{file_path}' not found. Starting with empty registry.")
        return {}

    try:
        with open(file_path, "r", encoding="utf-8") as file:
            data = json.load(file)
            print(f"[SUCCESS] Loaded {len(data)} record(s) from {file_path}.")
            return data
    except json.JSONDecodeError as json_err:
        print(f"[ERROR] Corrupted JSON structure in '{file_path}': {json_err}")
        return {}
    except Exception as err:
        print(f"[UNEXPECTED ERROR] Failed to load data: {err}")
        return {}


def view_all_records(registry: Dict[str, Dict[str, Any]]) -> None:
    """Prints all student records in a formatted tabular view."""
    if not registry:
        print("\n[INFO] No records found in the registry.")
        return

    separator = "-" * 75
    print("\n" + separator)
    print(f"{'Student ID':<12} | {'Name':<22} | {'Branch':<22} | {'CGPA':<5}")
    print(separator)
    for student_id, details in registry.items():
        name = details.get("name", "N/A")
        branch = details.get("branch", "N/A")
        cgpa = details.get("cgpa", 0.0)
        print(f"{student_id:<12} | {name:<22} | {branch:<22} | {cgpa:<5.2f}")
    print(separator + "\n")


#to do

def add_student_record(registry: Dict[str, Dict[str, Any]]) -> None:
    """TODO Task 1: Prompt user for student ID, name, branch, and CGPA.

    Validation Rules:
      1. Student ID must not already exist in the registry.
      2. Name and Branch must not be empty after stripping whitespace.
      3. CGPA must be a valid float between 0.0 and 10.0.
      4. Auto-generate email: <first_name_lowercase>.<id_lowercase>@university.edu
    """
    print("\n--- Add New Student ---")

    # Student ID
    student_id = input("Enter Student ID: ").strip()

    if student_id == "":
        print("[ERROR] Student ID cannot be empty.")
        return

    # Check if ID already exists
    if student_id in registry:
        print("[ERROR] Student ID already exists.")
        return

    # Student Name
    name = input("Enter Student Name: ").strip()

    if name == "":
        print("[ERROR] Name cannot be empty.")
        return

    # Branch
    branch = input("Enter Branch: ").strip()

    if branch == "":
        print("[ERROR] Branch cannot be empty.")
        return

    # CGPA
    cgpa_input = input("Enter CGPA (0-10): ").strip()

    try:
        cgpa = float(cgpa_input)
    except ValueError:
        print("[ERROR] CGPA must be a valid number.")
        return

    if cgpa < 0 or cgpa > 10:
        print("[ERROR] CGPA must be between 0.0 and 10.0.")
        return

    # Get first name
    first_name = name.split()[0].lower()

    # Generate email
    email = f"{first_name}.{student_id.lower()}@university.edu"

    # Add record to registry
    registry[student_id] = {
        "name": name,
        "branch": branch,
        "cgpa": cgpa,
        "email": email
    }

    print("[SUCCESS] Student record added successfully.")


def search_student_record(registry: Dict[str, Dict[str, Any]]) -> None:
    """TODO Task 2: Search by student ID (exact) or student Name (case-insensitive substring)."""
    print("\n--- Search Student Records ---")

    search_text = input("Enter Student ID or Name: ").strip()

    if search_text == "":
        print("[ERROR] Search value cannot be empty.")
        return

    # Search by exact Student ID
    if search_text in registry:
        details = registry[search_text]

        print("\nStudent Found:")
        print(f"Student ID : {search_text}")
        print(f"Name       : {details.get('name', 'N/A')}")
        print(f"Branch     : {details.get('branch', 'N/A')}")
        print(f"CGPA       : {details.get('cgpa', 0.0):.2f}")
        print(f"Email      : {details.get('email', 'N/A')}")
        return

    # Search by name substring
    found = False

    for student_id, details in registry.items():
        name = details.get("name", "")

        if search_text.lower() in name.lower():

            if not found:
                print("\nMatching Records:")
                print("-" * 75)

            print(f"Student ID : {student_id}")
            print(f"Name       : {name}")
            print(f"Branch     : {details.get('branch', 'N/A')}")
            print(f"CGPA       : {details.get('cgpa', 0.0):.2f}")
            print(f"Email      : {details.get('email', 'N/A')}")
            print("-" * 75)

            found = True

    if not found:
        print("[INFO] No matching student record found.")


def delete_student_record(registry: Dict[str, Dict[str, Any]]) -> None:
    """TODO Task 3: Prompt for Student ID and delete record with confirmation."""
    print("\n--- Delete Student Record ---")

    student_id = input("Enter Student ID to delete: ").strip()

    if student_id not in registry:
        print("[ERROR] Student ID not found.")
        return

    student_name = registry[student_id].get("name", "N/A")

    print(f"Student found: {student_name}")

    confirmation = input(
        "Are you sure you want to delete? (y/n): "
    ).strip().lower()

    if confirmation == "y":
        del registry[student_id]
        print("[SUCCESS] Student record deleted successfully.")
    else:
        print("[INFO] Delete operation cancelled.")


def update_student_record(registry: Dict[str, Dict[str, Any]]) -> None:
    """TODO Task 4: Update an existing student's name, branch, and CGPA."""
    print("\n--- Update Student Record ---")

    student_id = input("Enter Student ID to update: ").strip()

    if student_id not in registry:
        print("[ERROR] Student ID not found.")
        return

    student = registry[student_id]

    print("\nPress Enter to keep the old value.")

    # Update Name
    new_name = input(
        f"Enter Name [{student.get('name', '')}]: "
    ).strip()

    if new_name != "":
        student["name"] = new_name

        # Update email if name changes
        first_name = new_name.split()[0].lower()
        student["email"] = (
            f"{first_name}.{student_id.lower()}@university.edu"
        )

    # Update Branch
    new_branch = input(
        f"Enter Branch [{student.get('branch', '')}]: "
    ).strip()

    if new_branch != "":
        student["branch"] = new_branch

    # Update CGPA
    new_cgpa = input(
        f"Enter CGPA [{student.get('cgpa', 0.0)}]: "
    ).strip()

    if new_cgpa != "":
        try:
            cgpa = float(new_cgpa)

            if cgpa < 0 or cgpa > 10:
                print("[ERROR] CGPA must be between 0.0 and 10.0.")
                return

            student["cgpa"] = cgpa

        except ValueError:
            print("[ERROR] CGPA must be a valid number.")
            return

    print("[SUCCESS] Student record updated successfully.")


def save_records_to_json(
    file_path: str,
    registry: Dict[str, Dict[str, Any]]
) -> None:
    """TODO Task 4: Serialize the in-memory registry dictionary to the JSON file safely."""

    try:
        with open(file_path, "w", encoding="utf-8") as file:

            # Save registry into JSON file
            json.dump(registry, file, indent=2)

            # Make sure data is written safely
            file.flush()
            os.fsync(file.fileno())

        print(f"[SUCCESS] {len(registry)} record(s) saved to {file_path}.")

    except Exception as err:
        print(f"[ERROR] Failed to save records: {err}")


def export_to_csv(
    file_path: str,
    registry: Dict[str, Dict[str, Any]]
) -> None:
    """TODO Task 5 (Bonus): Export all student records to a CSV file."""

    import csv

    if not registry:
        print("[INFO] No records available to export.")
        return

    try:
        with open(
            file_path,
            "w",
            newline="",
            encoding="utf-8"
        ) as file:

            fieldnames = [
                "student_id",
                "name",
                "branch",
                "cgpa",
                "email"
            ]

            writer = csv.DictWriter(
                file,
                fieldnames=fieldnames
            )

            # Write headings
            writer.writeheader()

            # Write records
            for student_id, details in registry.items():

                writer.writerow({
                    "student_id": student_id,
                    "name": details.get("name", ""),
                    "branch": details.get("branch", ""),
                    "cgpa": details.get("cgpa", ""),
                    "email": details.get("email", "")
                })

        print(f"[SUCCESS] Records exported to {file_path}.")

    except Exception as err:
        print(f"[ERROR] Failed to export CSV: {err}")


def main_menu() -> None:
    """Main CLI control loop."""
    global STUDENT_REGISTRY
    STUDENT_REGISTRY = load_records_from_json(DATABASE_FILE)

    menu_banner = """
========================================
🎓 STUDENT RECORD MANAGEMENT SYSTEM
========================================
1. View All Records
2. Add Student Record
3. Search Record
4. Delete Record
5. Save Database to File
6. Export Records to CSV (Bonus)
7. Update Student Record
0. Save & Exit
========================================
"""

    while True:
        print(menu_banner)

        choice = input("Enter choice [0-7]: ").strip()

        if choice == "1":
            view_all_records(STUDENT_REGISTRY)

        elif choice == "2":
            add_student_record(STUDENT_REGISTRY)

        elif choice == "3":
            search_student_record(STUDENT_REGISTRY)

        elif choice == "4":
            delete_student_record(STUDENT_REGISTRY)

        elif choice == "5":
            save_records_to_json(
                DATABASE_FILE,
                STUDENT_REGISTRY
            )

        elif choice == "6":
            export_to_csv(
                "students_export.csv",
                STUDENT_REGISTRY
            )

        elif choice == "7":
            update_student_record(STUDENT_REGISTRY)

        elif choice == "0":
            save_records_to_json(
                DATABASE_FILE,
                STUDENT_REGISTRY
            )

            print("[INFO] Application closed successfully. Good bye!")
            sys.exit(0)

        else:
            print(
                "[WARN] Invalid option selected. "
                "Please enter a number between 0 and 7."
            )


if __name__ == "__main__":
    main_menu()


[WARN] Database file 'sample_records.json' not found. Starting with empty registry.

🎓 STUDENT RECORD MANAGEMENT SYSTEM
1. View All Records
2. Add Student Record
3. Search Record
4. Delete Record
5. Save Database to File
6. Export Records to CSV (Bonus)
7. Update Student Record
0. Save & Exit

Enter choice [0-7]: 2

--- Add New Student ---
Enter Student ID: 2023322829
Enter Student Name: utkarsh gupta
Enter Branch: cse
Enter CGPA (0-10): 7
[SUCCESS] Student record added successfully.

🎓 STUDENT RECORD MANAGEMENT SYSTEM
1. View All Records
2. Add Student Record
3. Search Record
4. Delete Record
5. Save Database to File
6. Export Records to CSV (Bonus)
7. Update Student Record
0. Save & Exit

Enter choice [0-7]: 1

---------------------------------------------------------------------------
Student ID   | Name                   | Branch                 | CGPA 
---------------------------------------------------------------------------
2023322829   | utkarsh gupta          | cse           